# 03 — MuJoCoのGo2 Plant

制御器の外側にある、状態を生成しトルクを受け取る物理系を確認します。

**前提**: `02_vectors_units_and_frames.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 完全モデルと縮約モデル

MuJoCoは浮遊基部と12関節を持つ全身モデルを積分します。
一方MPCは胴体を単一剛体として近似します。

\[
q\in\mathbb{R}^{19},\quad \dot q\in\mathbb{R}^{18},\quad
\tau\in\mathbb{R}^{12}
\]

quaternionが4要素なので、浮遊基部の `nq` と `nv` は1だけ異なります。

In [2]:
# 背景: MuJoCoの浮遊基部モデルでは姿勢quaternionが4要素、角速度が3要素なので、一般化位置nqと速度nvの次元が一致しない。
# 目的: Go2環境を平坦面で初期化し、Plantのnq=19・nv=18・nu=12と実データshapeを直接検証する。
# 数値配列型を依存先が利用するためNumPyを読み込む。
import numpy as np
# MuJoCo四足環境を生成してモデル次元と状態バッファを調べるためQuadrupedEnvを読み込む。
from gym_quadruped.quadruped_env import QuadrupedEnv
# 教材と上流実装で同じrobot名・simulation刻み[s]を使うため設定を読み込む。
from quadruped_pympc import config as cfg

# 外乱の少ないflat sceneで、Go2全身Plantの次元だけを再現可能に調べる環境を構成する。
env = QuadrupedEnv(
    robot=cfg.robot, scene="flat", sim_dt=cfg.simulation_params["dt"],  # 上流と同じrobotおよび積分刻み[s]を使う。
    ref_base_lin_vel=0.0, ref_base_ang_vel=0.0,  # 次元確認に運動指令は不要なので並進・角速度参照を0にする。
    ground_friction_coeff=0.8, base_vel_command_type="forward",  # 標準的な摩擦係数と前進指令形式でPlantを構成する。
    state_obs_names=(),  # 観測ベクトルは使わないため空にし、モデル内部量だけを確認する。
)
# 乱数初期化を無効にして、同じ初期姿勢・shapeを再現可能に生成する。
env.reset(random=False)
# 一般化位置・速度・actuator入力の次元を並べ、q∈R^19, qdot∈R^18, tau∈R^12を確認する。
print("nq, nv, nu:", env.mjModel.nq, env.mjModel.nv, env.mjModel.nu)
# quaternionを含む一般化位置バッファがshape (19,)であることを表示する。
print("qpos shape:", env.mjData.qpos.shape)
# 浮遊基部6速度と12関節速度からなるバッファがshape (18,)であることを表示する。
print("qvel shape:", env.mjData.qvel.shape)
# 想定モデル以外を誤って読み込んだ場合に後続の次元議論を止める。
assert (env.mjModel.nq, env.mjModel.nv, env.mjModel.nu) == (19, 18, 12)
# MuJoCo資源を明示解放し、Notebook再実行時に不要な環境を残さない。
env.close()

nq, nv, nu: 19 18 12
qpos shape: (19,)
qvel shape: (18,)


/home/takuya/work/mpc_dog/.venv/lib/python3.11/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/takuya/work/mpc_dog/.venv/lib/python3.11/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## インターフェース

`simulation.py` はworld座標の足位置・COM速度、base座標の角速度、
Jacobian、質量行列などを集めて `compute_actions` に渡します。
戻った脚別トルクをactuator順へ詰め、上限の90%でclipして `env.step(action)` します。

MPCの予測が正しくても、最終トルクclipが頻発すれば実機Plantは予測通り動きません。
したがって後の診断では「目標GRF」「変換後トルク」「clip後トルク」を分けます。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。